# **Naive estimator**

#### Imports

In [37]:
import numpy as np

#### Calibration

In [38]:
def rho_from_pc(pc, lam):
    return lam * pc / (1.0 - pc)

def cmax_from_pc(pc, lam):
    h = lambda c: (1.0 - np.exp(-lam * c)) / (lam * c)

    lo, hi = 1.0, 2.0
    while h(hi / lam) > pc:
        hi *= 2.0
    for _ in range(300):
        mid = 0.5 * (lo + hi)
        if h(mid / lam) > pc:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi) / lam

#### Simulation

In [39]:
def get_y_delta(scheme, pc, n, reps, seed, lam):
    rng = np.random.default_rng(seed)
    param = cmax_from_pc(pc, lam) if scheme == "uniform" else rho_from_pc(pc, lam)

    t = rng.exponential(scale=1.0 / lam, size=(reps, n))
    if pc <= 0:
        c = np.full((reps, n), np.inf)
    elif scheme == "uniform":
        c = rng.uniform(0.0, param, size=(reps, n))
    elif scheme == "exponential":
        c = rng.exponential(scale=1.0 / param, size=(reps, n))
    else:
        raise ValueError(scheme)

    y = np.minimum(t, c)
    delta = t <= c
    return y, delta

#### Estimate

In [40]:
def naive_estimate(y):
    return len(y) / y.sum()

def mle_estimate(y, delta):
    r = int(delta.sum())
    if r == 0:
        return np.nan
    return r / y.sum()

#### Monte Carlo

In [41]:
def mc_summary(est, lam):
    est = np.asarray(est, float)
    est = est[np.isfinite(est)]
    err = est - lam
    return {
        "n_used": est.size,
        "bias": float(est.mean() - lam),
        "se_bias": float(est.std(ddof=0) / np.sqrt(est.size)),
        "var": float(est.var(ddof=0)),
        "mse": float((err ** 2).mean()),
        "se_mse": float((err ** 2).std(ddof=0) / np.sqrt(est.size)),
    }


def mc_cell(n, scheme, pc, reps, seed, lam):
    y, delta = get_y_delta(scheme, pc, n, reps, seed, lam)

    r = delta.sum(axis=1)

    naive = np.empty(reps, float)
    mle = np.empty(reps, float)
    for i in range(reps):
        naive[i] = naive_estimate(y[i]); mle[i] = mle_estimate(y[i], delta[i])

    return {"naive": naive, "mle": mle,
            "obs_pc": 1.0 - r.mean() / n,
            "r0_share": float((r == 0).mean())}

def run_sim(seed, n, reps, lam, schemes=("uniform", "exponential")):
    for scheme in schemes:
        rows = []
        for pc in (0.0, 0.10, 0.50):
            cell = mc_cell(n, scheme, pc, reps, seed=seed, lam=lam)
            rows.append((pc, cell,
                         mc_summary(cell["naive"], lam),
                         mc_summary(cell["mle"], lam)))

        print(f"\n{scheme} censoring")
        print(f"{'pc input':>14} {'pc obs':>8} {'r=0':>6} "
              f"{'naive bias':>8} {'nv var':>9} {'nv Used':>8} {'nv MSE':>11} {'nv SE':>9} "
              f"{'MLE':>8} {'ml var':>9} {'ml Used':>8} {'ml MSE':>11} {'ml SE':>9} "
              f"{'ratio':>7}")
        print("-" * 148)
        for pc, cell, nv, ml in rows:
            print(f"{f'{pc:.0%}':>14} {cell['obs_pc']:>8.2%} "
                  f"{cell['r0_share']:>6.2%} "
                  f"{nv['bias']:>8.5f} {nv['var']:>9.2e} {nv['n_used']:>8d} "
                  f"{nv['mse']:>11.3e} {nv['se_mse']:>9.2e} "
                  f"{ml['bias']:>8.5f} {ml['var']:>9.2e} {ml['n_used']:>8d} "
                  f"{ml['mse']:>11.3e} {ml['se_mse']:>9.2e} "
                  f"{nv['mse'] / ml['mse']:>7.1f}")


In [42]:
lambda_ = 0.05
n = 500
reps = 2000
seed = 42

run_sim(seed, n, reps, lambda_)


uniform censoring
      pc input   pc obs    r=0 naive bias    nv var  nv Used      nv MSE     nv SE      MLE    ml var  ml Used      ml MSE     ml SE   ratio
----------------------------------------------------------------------------------------------------------------------------------------------------
            0%    0.00%  0.00%  0.00011  4.83e-06     2000   4.843e-06  1.58e-07  0.00011  4.83e-06     2000   4.843e-06  1.58e-07     1.0
           10%   10.00%  0.00%  0.00565  5.85e-06     2000   3.782e-05  6.56e-07  0.00009  5.37e-06     2000   5.382e-06  1.70e-07     7.0
           50%   49.99%  0.00%  0.05016  1.23e-05     2000   2.528e-03  7.95e-06  0.00010  9.90e-06     2000   9.907e-06  3.09e-07   255.2

exponential censoring
      pc input   pc obs    r=0 naive bias    nv var  nv Used      nv MSE     nv SE      MLE    ml var  ml Used      ml MSE     ml SE   ratio
--------------------------------------------------------------------------------------------------------------

### Metrics

pc input = percentage of censoring \
pc obs = actual data censored \
r = pc of runs with no events happening \
naive bias = naive bias \
nv var = naive variance \
nv Used = number of runs used for the naive \
nv MSE = MSE of Naive Estimator \
nv SE = SE of Naive Estimator \
Same for MLE \
ratio = nv MSE / ml MSE